In [82]:
import sys
import os
import pandas as pd 
sys.path.append(os.path.abspath(".."))
import torch
from src.infrence.model import load_model
from src.infrence.generate import generate_text
import json
from pydantic import BaseModel
from typing import Optional, Literal
import validation


ModuleNotFoundError: No module named 'validation'

In [2]:
tokenizer, model = load_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.59k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [22]:
# CLASSIFICATION TICKET 
prompt = """ 
You are expert ticket classifier. 
your task is to classify user ticket and generate valid json file only.
Only valis json is accepted, no need to extra text , no need for explanation.
Required fields:
- category
- sentiment
- urgency
- summary

Allowed category values:
technical, account, delivery, billing, subscription

Allowed sentiment values:
positive, negative, neutral

Allowed urgency values:
low, medium, high
output_format : 
{{
"category" : <"technical", "account", "delivery", "billing", "subscription">,
"sentiment": <"positive", "negative", "neutral">,
"urgency"  : <"low", "medium", "high">,
"summary"  : "only one or two sentence describe user query"
}}

user_query :
{message}
"""



In [23]:
tickets = pd.read_csv('data/tickets.csv')
tickets.iloc[3,1]

'I was charged twice for the same order.'

In [24]:
prompt = prompt.format(message=tickets.iloc[3,1])

In [25]:
output = generate_text(
    tokenizer=tokenizer,
    model = model,
    prompt=prompt,
    top_p=.9,
    temperature=.2
)

print("\n --- generated output ---")
print(output)


 --- generated output ---
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want to cancel the order.
I want


In [12]:
type(output)

str

In [54]:
data = '''{
  "category": "charge",
  "sentiment": "charge",
  "urgency": "urgent",
  "summary": "I was charged twice for the same order."
}'''

In [21]:
data = json.loads(data)
print(data)
print(type(data))

{'category': 'charge', 'subscription': 'charge', 'urgency': 'urgent', 'summary': 'I was charged twice for the same order.'}
<class 'dict'>


In [ ]:
# # SCHEMA

# class ww(BaseModel) :
#     urgency_level : str 
#     urgency_code : int


# class TicketOutput(BaseModel) : 
#     category : str 
#     urgency  : ww  
#     summary  : str
#     sentiment: Optional[str] = None,
#     history  : list[str]
#     meta_data: dict[str, str]

In [ ]:
# TicketOutput(
#     category="adasd",
#     summary = "Asda",
#     urgency= {
#         'urgency_level' : 'low',
#         'urgency_code' :1001
#     }
    
# )

TicketOutput(category='adasd', urgency=ww(urgency_level='low', urgency_code=1001), summary='Asda', sentiment=None)

In [84]:
data = '''{
  "category": "technical",
  "sentiment": "positive",
  "urgency": "3",
  "summary": "I was charged twice for the same order."
}'''
# data = json.loads(data)
# TicketOutput.model_validate(data)

In [86]:
parse_and_validate_ticket(data)

NameError: name 'validation' is not defined